# First look: sleep and recovery, summer vs. term

Built from `data/health.duckdb`. Run `python -m recovery` first if it's missing.

This is one person's data, so everything here describes me, not people in general. Summer was in Taiwan and the term is in the US, so the summer vs. term comparison also includes a change of place and time zone.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from recovery.analysis import compare, exam_changes, exam_summary, load_daily, study_days, travel_days
from recovery.model import evaluate

# Reference palette slots 1 and 2. Blue is the US and orange is Taiwan everywhere.
BLUE, ORANGE = "#2a78d6", "#eb6834"
INK, INK_2, MUTED, GRID, SURFACE, BAND = "#0b0b0b", "#52514e", "#b5b4ae", "#e6e5e1", "#fcfcfb", "#f1efe9"

plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
    "axes.edgecolor": GRID, "axes.labelcolor": INK_2, "axes.titlecolor": INK,
    "axes.titlesize": 11, "axes.titleweight": "bold", "axes.titlelocation": "left",
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "axes.grid.axis": "y", "grid.color": GRID, "grid.linewidth": 0.8,
    "xtick.color": INK_2, "ytick.color": INK_2, "font.size": 10, "lines.linewidth": 2,
    "legend.frameon": False,
})
pd.set_option("display.precision", 2)

# Bedtimes run past midnight, so they're stored as 25 = 1am. Show them as clock times.
NIGHT = (21, 31)


def clock_axis(axis, step=2):
    hours = range(NIGHT[0] + 1, NIGHT[1], step)
    axis.set_ticks(hours, [f"{h % 24}:00" for h in hours])

daily = load_daily()
print(f"{len(daily)} days, {daily.date.min():%Y-%m-%d} to {daily.date.max():%Y-%m-%d}")

## 1. What data exists, and when

Each tick is a day with a value. The Apple Watch was replaced by a Fitbit in June. The two measure HRV differently, so the rest of the notebook uses only the Fitbit period.

In [ ]:
coverage = {
    "Sleep": "main_sleep_minutes", "Sleep score": "sleep_score", "Resting HR": "resting_hr",
    "HRV": "hrv_ms", "Readiness": "readiness_score", "Steps": "steps",
}
fitbit_start = daily.loc[daily.heart_device == "fitbit", "date"].min()

fig, ax = plt.subplots(figsize=(10, 2.8))
for row, (name, column) in enumerate(coverage.items()):
    days = daily.loc[daily[column].notna(), "date"]
    ax.vlines(days, row - 0.3, row + 0.3, color=BLUE, linewidth=1.5)
ax.set_yticks(range(len(coverage)), coverage.keys())
ax.invert_yaxis()
ax.grid(False)
ax.axvline(fitbit_start, color=INK_2, linewidth=1, linestyle="--")
ax.text(fitbit_start, -0.8, "  Fitbit from here", color=INK_2, va="bottom")
ax.set_title("Days with data")
plt.show()

## 2. Where I was

A day's location comes from the time zone saved with that night's sleep. The first week after a long flight is marked as jet lag, and periods marked as trips in `calendar.csv` are left out too, so each group is days at home.

In [ ]:
trips = travel_days(daily)
display(trips)

era = study_days(daily, jet_lag_days=7)
summer, term = era["summer"], era["term"]
print(f"Summer in Taiwan: {summer.sum()} days. Term in the US since {era.loc[term, 'date'].min():%b %d}: {term.sum()} days.")

## 3. Trends

Grey dots are single days and the line is a 7-day average. Shaded stretches are Taiwan. The bedtime panel shows 9pm to 7am, so the two flight days, when the main sleep was at an odd hour, fall off it.

In [ ]:
trend = {
    "sleep_hours": "Sleep (hours)", "bedtime_hour": "Bedtime",
    "resting_hr": "Resting HR (bpm)", "hrv_ms": "HRV (ms)",
    "readiness_score": "Readiness", "steps": "Steps",
}
taiwan_runs = (era["location"] == "Taiwan").astype(int).diff().fillna(0).ne(0).cumsum()
spans = era[era["location"] == "Taiwan"].groupby(taiwan_runs)["date"].agg(["min", "max"])

fig, axes = plt.subplots(len(trend), 1, figsize=(10, 13), sharex=True)
for ax, (column, name) in zip(axes, trend.items()):
    for _, span in spans.iterrows():
        ax.axvspan(span["min"], span["max"] + pd.Timedelta(days=1), color=BAND, zorder=0, linewidth=0)
    ax.scatter(era["date"], era[column], s=10, color=MUTED, zorder=1)
    ax.plot(era["date"], era[column].rolling(7, min_periods=4, center=True).mean(), color=BLUE, zorder=2)
    ax.set_title(name)
    if column == "bedtime_hour":
        ax.set_ylim(*NIGHT)
        clock_axis(ax.yaxis)
for _, span in spans.iterrows():
    axes[0].text(span["min"] + (span["max"] - span["min"]) / 2, 0.95, "Taiwan",
                 transform=axes[0].get_xaxis_transform(), ha="center", va="top", color=INK_2)
fig.tight_layout()
plt.show()

## 4. Summer in Taiwan vs. term in the US

The interval is a 95% block bootstrap. It resamples whole weeks, because neighboring days are alike and treating each day as independent would make the interval too narrow. An interval that doesn't include 0 is a difference worth taking seriously. With eight measures, though, one of them could cross that line by chance.

In [ ]:
compare(era, term, summer, "term (US)", "summer (Taiwan)")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.5))
bins = np.arange(22, 30.5, 0.5)
for mask, color, label in [(summer, ORANGE, "Summer (Taiwan)"), (term, BLUE, "Term (US)")]:
    bedtimes = era.loc[mask, "bedtime_hour"].dropna()
    ax.hist(bedtimes, bins=bins, histtype="step", linewidth=2, color=color, label=label,
            weights=np.full(len(bedtimes), 1 / len(bedtimes)))
ax.set_xlim(*NIGHT)
clock_axis(ax.xaxis, step=1)
ax.set_ylabel("share of nights")
ax.legend(loc="upper right")
ax.set_title("When I fall asleep")
plt.show()

## 5. Internship vs. free summer days

Both groups are days in Taiwan, so place and time zone are the same. Internship days are labeled `internship` in `calendar.csv`, and the free days are the rest of the summer at home.

In [ ]:
internship = summer & (era["period"] == "internship")
free_summer = summer & (era["period"] == "summer")
compare(era, internship, free_summer, "internship", "free summer")

## 6. Weekdays vs. weekends during the term

This compares days in the same place and time zone, so it's a cleaner test of a schedule effect than summer vs. term. A day here is the night before it, so Saturday and Sunday mornings come after Friday and Saturday nights.

In [ ]:
weekend = era["is_weekend"].astype(bool)
compare(era, term & weekend, term & ~weekend, "term weekends", "term weekdays")

## 7. Jet lag

Bedtime around each long flight, with day 0 as the first day in the new time zone. The night of the flight itself is usually off the chart.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.5))
for _, trip in trips.iterrows():
    if trip["date"] < fitbit_start - pd.Timedelta(days=7):
        continue
    window = daily[daily["date"].between(trip["date"] - pd.Timedelta(days=5), trip["date"] + pd.Timedelta(days=14))]
    offset = (window["date"] - trip["date"]).dt.days
    color = ORANGE if trip["to"] == "Taiwan" else BLUE
    label = f"To {trip['to']}, {trip['date']:%b %d}"
    ax.plot(offset, window["bedtime_hour"], color=color, marker="o", markersize=6,
            markeredgecolor=SURFACE, markeredgewidth=1.5, label=label)
ax.axvline(0, color=INK_2, linewidth=1, linestyle="--")
ax.set_xlabel("days since arrival")
ax.set_xticks(range(-5, 15, 1))
ax.set_ylim(*NIGHT)
clock_axis(ax.yaxis)
ax.legend(loc="upper right")
ax.set_title("Bedtime around long flights")
plt.show()

## 8. How sleep relates to the next day

Fitbit's readiness score uses sleep, HRV, and resting heart rate as inputs, so some link between sleep and readiness is built into the score. HRV is a cleaner thing to check against.

In [ ]:
settled = era[~era["jet_lag"]]
pairs = [("sleep_hours", "hrv_ms", "Sleep (hours)", "HRV that morning (ms)"),
         ("bedtime_hour", "hrv_ms", "Bedtime", "HRV that morning (ms)")]

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, (x, y, x_name, y_name) in zip(axes, pairs):
    both = settled[[x, y, "location"]].dropna()
    colors = np.where(both["location"] == "Taiwan", ORANGE, BLUE)
    ax.scatter(both[x], both[y], s=36, c=colors, edgecolors=SURFACE, linewidths=1.5)
    rho = both[x].rank().corr(both[y].rank())  # Spearman: correlation of ranks
    ax.set_xlabel(x_name)
    ax.set_ylabel(y_name)
    ax.set_title(f"Spearman correlation {rho:.2f} ({len(both)} days)")
    if x == "bedtime_hour":
        clock_axis(ax.xaxis, step=1)
axes[1].scatter([], [], s=36, c=ORANGE, label="Taiwan")
axes[1].scatter([], [], s=36, c=BLUE, label="US")
axes[1].legend(loc="upper right")
fig.tight_layout()
plt.show()

## 9. Exams

Each exam is compared with its own baseline: the 7 most recent ordinary days ending 3 days before it, measured by the same device, skipping other exams, trips, and jet lag. A day's row holds the night before it, so day 0 is the night before the exam. Comparing each exam with its own recent past keeps the Apple Watch to Fitbit change and the seasons out of the result, so spring exams from the Apple Watch period count too. Sleep data starts in late May, so only the June and October exams have sleep.

Exams less than a week apart share a stretch of days, so the interval resamples whole exam periods, and it's left blank with fewer than 4 periods. HRV changes mix two devices that measure HRV differently (Apple's SDNN before June, Fitbit's RMSSD after), so treat the HRV row loosely. Resting heart rate and steps compare fairly across the two.

In [ ]:
changes = exam_changes(daily)
exam_summary(changes)

In [ ]:
panels = {"resting_hr": "Resting HR (bpm)", "hrv_ms": "HRV (ms)", "sleep_hours": "Sleep (hours)", "steps": "Steps"}
fig, axes = plt.subplots(2, 2, figsize=(10, 6.5), sharex=True)
for ax, (column, name) in zip(axes.flat, panels.items()):
    for _, one in changes.dropna(subset=[column]).groupby("exam"):
        ax.plot(one["offset"], one[column], color=MUTED, linewidth=1)
    average = changes.groupby("offset")[column].mean()
    ax.plot(average.index, average, color=BLUE, marker="o", markersize=6, markeredgecolor=SURFACE, markeredgewidth=1.5)
    ax.axhline(0, color=INK_2, linewidth=1)
    ax.axvline(0, color=INK_2, linewidth=1, linestyle="--")
    ax.set_title(name)
    ax.set_xticks(range(-7, 4))
for ax in axes[1]:
    ax.set_xlabel("days from exam (0 = the night before)")
fig.suptitle("Change from each exam's baseline. Grey lines are single exams, blue is the average.",
             x=0.01, ha="left", color=INK_2)
fig.tight_layout()
plt.show()

## 10. Can tomorrow's HRV be predicted?

Two ridge regression models are compared with two simple guesses. The evening model uses only what's known before going to sleep: recent HRV and resting heart rate, yesterday's steps and active minutes, last night's sleep, weekday or weekend, and location. The morning model adds the night's sleep itself. The predictions are walk-forward: each day is predicted by a model trained only on the days before it, starting after 42 days.

"vs. 7-day average" is the change in error compared with the 7-day average, so negative is better. The interval is a block bootstrap over the test days.

In [ ]:
predictions, results = evaluate(daily, "hrv_ms")
print(f"{results.attrs['days']} test days, {predictions.date.min():%b %d} to {predictions.date.max():%b %d}")
results

In [ ]:
# One row per calendar day, so lines break over the gaps instead of bridging them.
shown = predictions.set_index("date").asfreq("D")
fig, ax = plt.subplots(figsize=(10, 3.5))
ax.scatter(shown.index, shown["actual"], s=12, color=MUTED, label="Actual HRV")
ax.plot(shown.index, shown["week_mean"], color=ORANGE, label="Average of last 7 days")
ax.plot(shown.index, shown["evening"], color=BLUE, label="Evening model")
ax.set_ylabel("HRV (ms)")
ax.legend(loc="lower left", ncols=3)
ax.set_title("Predicted vs. actual HRV on the test days")
plt.show()

In [ ]:
_, readiness_results = evaluate(daily, "readiness_score")
readiness_results

## Notes

What stands out, and what to check next:

-